# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question (same as Weeks 2–4):** which pages should an editor look at first because they are about to lose search impressions? It's a *"which first?"* ranking problem with an observed yes/no label (`is_declining`), so I need a classifier's **probability** as the ranking score, evaluated at **Precision@K**.

**Methods I compare, and why each one is here:**
- **Logistic Regression.** The simplest readable model. If it already wins, I don't need anything heavier.
- **Decision Tree (depth 3).** Close to a hand-written rule, but learned from data. I can print it and read it like my baseline.
- **Random Forest.** Handles non-linear patterns (like the age signal that goes up and then down in Week 4) without tuning. This is my main candidate.
- **Gradient Boosting (HistGradientBoosting).** Only included as a "does more complexity help?" check. It's safe here because the inputs are the same leak-checked features.

**Features (7, all knowable on 15 March):** the five from my Week 3 contract (`log_imp_h1`, `ctr_h1`, `avg_pos_h1`, `active_days_h1`, `momentum_h1`) plus the two Week 4 signals (`age_days`, `ctr_gap`). No future-window or label-derived column goes in.

**Choosing a winner:** I don't pick the most complex model by default. A model has to beat the baseline *and* earn its complexity against the simpler ones.

In [1]:
import sys
sys.path.insert(0, "../scripts")
import json
import numpy as np
import pandas as pd
import sklearn
import lane_data as L
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

con = L.connect()
frame = L.baseline_score(L.build_frame(con))   # same rows + the frozen Week-4 baseline score

FEATURES = ["log_imp_h1", "ctr_h1", "avg_pos_h1", "active_days_h1", "momentum_h1", "age_days", "ctr_gap"]
X_all = frame[FEATURES].copy()
X_all["avg_pos_h1"] = X_all["avg_pos_h1"].fillna(100)   # 3 pages had no recorded position -> treat as "very deep"
X_all["ctr_gap"] = X_all["ctr_gap"].fillna(0)
X_all["momentum_h1"] = X_all["momentum_h1"].clip(upper=10)  # a few pages spike 100x+; cap so outliers don't dominate
frame[FEATURES] = X_all

def make_models():
    return {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "Decision Tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=L.SEED),
        "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=L.SEED, n_jobs=-1),
        "Gradient Boosting": HistGradientBoostingClassifier(random_state=L.SEED),
    }

print(f"Pages: {len(frame):,} | clients: {frame['client_hash_id'].nunique()} | seed: {L.SEED} | scikit-learn {sklearn.__version__}")

C:\Users\Itachi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages: 92,548 | clients: 40 | seed: 42 | scikit-learn 1.9.0


The learned tree's first three splits are **all on momentum**, then age (older than ~174 days → safer, the same effect my Week 4 age table showed) and activity/position. My hand-written rule never looked at momentum, which is why it lost. The age signal I found in Week 4 does show up here, but the CTR-gap signal barely matters once momentum is known.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Main split: the exact same client-grouped holdout as my Week 4 baseline** (`L.client_split`: 25% of clients held out, seed 42). Every page from a test client is unseen during training. This is the honest version of my question: *"will this work for a client whose pages it has never seen?"* A random row split would leak each client's style (publishing batches, site strength) from train into test.

**Time-aware by design:** all features come from 1–15 March and the label from 16–31 March, so the model never sees the future window.

**Stability check:** one holdout can be lucky, so I also run **5-fold GroupKFold by client** and report the mean and spread.

In [2]:
from sklearn.model_selection import GroupKFold

train, test = L.client_split(frame)
overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"Train: {len(train):,} pages / {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} pages / {test['client_hash_id'].nunique()} clients")
print(f"Clients in both train and test: {len(overlap)}")
print(f"Base rate  train={train['is_declining'].mean():.3f}  test={test['is_declining'].mean():.3f}")

Train: 67,662 pages / 30 clients
Test:  24,886 pages / 10 clients
Clients in both train and test: 0
Base rate  train=0.298  test=0.405


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
def evaluate(scores, labels):
    return {"P@50": L.precision_at_k(scores, labels, 50),
            "P@100": L.precision_at_k(scores, labels, 100),
            "P@500": L.precision_at_k(scores, labels, 500),
            "ROC AUC": roc_auc_score(labels, scores)}

y_tr, y_te = train["is_declining"], test["is_declining"]
rows = [{"method": "Random picking (base rate)", **{k: y_te.mean() for k in ["P@50", "P@100", "P@500"]}, "ROC AUC": 0.5},
        {"method": "Week-4 baseline rule", **evaluate(test["baseline_score"], y_te)}]

fitted, test_scores = {}, {}
for name, model in make_models().items():
    model.fit(train[FEATURES], y_tr)
    fitted[name] = model
    test_scores[name] = model.predict_proba(test[FEATURES])[:, 1]
    rows.append({"method": name, **evaluate(test_scores[name], y_te)})

# Sanity check against complexity: rank by ONE feature, no model at all (lower momentum = more likely to decline)
rows.append({"method": "Momentum only (1 feature, no model)", **evaluate(-test["momentum_h1"], y_te)})

holdout_table = pd.DataFrame(rows).set_index("method").round(3)
print("Same split, same label, same metrics (test clients only):")
holdout_table

Same split, same label, same metrics (test clients only):


,P@50,P@100,P@500,ROC AUC
method,,,,
Random picking (base rate),0.405,0.405,0.405,0.500
Week-4 baseline rule,0.380,0.440,0.438,0.589
Logistic Regression,0.880,0.820,0.760,0.721
Decision Tree (depth 3),0.940,0.930,0.902,0.710
Random Forest,0.940,0.930,0.890,0.760
Gradient Boosting,0.980,0.950,0.892,0.751
"Momentum only (1 feature, no model)",0.940,0.960,0.898,0.756


In [4]:
cv_rows = []
for fold, (a, b) in enumerate(GroupKFold(n_splits=5).split(frame, groups=frame["client_hash_id"])):
    tr, te = frame.iloc[a], frame.iloc[b]
    cv_rows.append({"fold": fold, "method": "Week-4 baseline rule", **evaluate(te["baseline_score"], te["is_declining"])})
    cv_rows.append({"fold": fold, "method": "Momentum only (1 feature, no model)", **evaluate(-te["momentum_h1"], te["is_declining"])})
    for name, model in make_models().items():
        model.fit(tr[FEATURES], tr["is_declining"])
        cv_rows.append({"fold": fold, "method": name, **evaluate(model.predict_proba(te[FEATURES])[:, 1], te["is_declining"])})

cv = pd.DataFrame(cv_rows)
cv_table = cv.groupby("method")[["P@50", "P@500", "ROC AUC"]].agg(["mean", "std"]).round(3)
print("5-fold GroupKFold by client (mean and std across folds):")
cv_table

5-fold GroupKFold by client (mean and std across folds):


P@50         P@500        ROC AUC       
                                      mean    std   mean    std    mean    std
method                                                                        
Decision Tree (depth 3)              0.756  0.213  0.714  0.105   0.649  0.040
Gradient Boosting                    0.916  0.030  0.798  0.059   0.677  0.045
Logistic Regression                  0.868  0.036  0.728  0.104   0.655  0.053
Momentum only (1 feature, no model)  0.932  0.027  0.790  0.062   0.669  0.047
Random Forest                        0.884  0.030  0.789  0.066   0.690  0.041
Week-4 baseline rule                 0.392  0.023  0.364  0.062   0.573  0.015

**Reading the comparison:**
- **Every learned model beats the Week-4 rule by a wide margin**, on the fixed holdout and in all five client folds. The rule's P@50 stays around 0.38–0.39, *below* random. The models average about 0.87–0.92 across folds.
- **The big surprise: one feature with no model is just as good at the top of the list.** Ranking by `momentum_h1` alone gets 5-fold P@50 = **0.932**, better than the Random Forest (0.884) and Gradient Boosting (0.916). At P@500 they're tied (~0.79). The models only add value across the **whole** list: RF ROC AUC 0.690 vs 0.669 for momentum alone. Almost all the win comes from one fact: *pages already losing impressions in week 2 keep losing them.*
- **My choice, without rewarding complexity:**
  - For the editor's weekly **top-50 queue** (the metric I committed to in Week 2), I'd use the **momentum ranking**. The models don't beat it there, so they haven't earned the extra complexity.
  - For scoring the **full list** (e.g. the next 500–5,000 pages, or sorting within a client), I keep the **Random Forest**. It has the best ROC AUC on the holdout and in every fold summary, it's stable (P@50 std 0.03), and it needs no tuning. Gradient Boosting is within one std of it everywhere, so its small P@50 edge isn't a reliable difference.
  - From now on, **momentum-only becomes the honest baseline** any later model has to beat.
- The holdout numbers are higher than the 5-fold means because the holdout test clients have a higher base rate (0.405 vs 0.298 in train). **The 5-fold means are my headline numbers.**

In [5]:
with open("../outputs/w05_model_metrics.json", "w") as fh:
    json.dump({"split": "GroupShuffleSplit by client, test_size=0.25, seed=42 (same as Week 4)",
               "features": FEATURES, "sklearn": sklearn.__version__,
               "holdout": holdout_table.reset_index().to_dict(orient="records"),
               "groupkfold_5": cv.groupby("method")[["P@50", "P@500", "ROC AUC"]].mean().round(3).reset_index().to_dict(orient="records")},
              fh, indent=2)
print("Saved work/outputs/w05_model_metrics.json")

Saved work/outputs/w05_model_metrics.json


**The readable model, printed.** This is the depth-3 tree, i.e. what a data-learned rule looks like next to my hand-written one:

In [6]:
print(export_text(fitted["Decision Tree (depth 3)"], feature_names=FEATURES, decimals=2))

|--- momentum_h1 <= 0.68
|   |--- momentum_h1 <= 0.32
|   |   |--- momentum_h1 <= 0.17
|   |   |   |--- class: 1
|   |   |--- momentum_h1 >  0.17
|   |   |   |--- class: 1
|   |--- momentum_h1 >  0.32
|   |   |--- age_days <= 173.50
|   |   |   |--- class: 1
|   |   |--- age_days >  173.50
|   |   |   |--- class: 0
|--- momentum_h1 >  0.68
|   |--- active_days_h1 <= 13.50
|   |   |--- ctr_gap <= -0.00
|   |   |   |--- class: 0
|   |   |--- ctr_gap >  -0.00
|   |   |   |--- class: 0
|   |--- active_days_h1 >  13.50
|   |   |--- avg_pos_h1 <= 22.12
|   |   |   |--- class: 0
|   |   |--- avg_pos_h1 >  22.12
|   |   |   |--- class: 0



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### What does the Random Forest lean on? (permutation importance on the test clients)
Each feature is shuffled in turn to see how much the test ROC AUC drops. A bigger drop means the model relies on that feature more.

In [7]:
from sklearn.inspection import permutation_importance

rf = fitted["Random Forest"]
pi = permutation_importance(rf, test[FEATURES], y_te, scoring="roc_auc", n_repeats=5, random_state=L.SEED, n_jobs=-1)
importance = (pd.DataFrame({"feature": FEATURES, "auc_drop_mean": pi.importances_mean, "auc_drop_std": pi.importances_std})
                .sort_values("auc_drop_mean", ascending=False).round(4))
importance

,feature,auc_drop_mean,auc_drop_std
4,momentum_h1,0.2074,0.0021
0,log_imp_h1,0.0190,0.0009
1,ctr_h1,0.0126,0.0011
5,age_days,0.0106,0.0013
3,active_days_h1,0.0085,0.0007
2,avg_pos_h1,0.0064,0.0003
6,ctr_gap,-0.0020,0.0009


**Top 3 features and why they plausibly matter:**
1. **`momentum_h1`** (by far the biggest). A page already losing impressions in week 2 is mid-slide, and slides usually continue for a while. Plausible, and not leakage: it only uses 1–15 March.
2. **`log_imp_h1`.** Bigger pages have steadier traffic, so random noise is less likely to push them across the 80% line.
3. **`ctr_h1`** / **`age_days`** (almost tied). Low-CTR and "settling" pages drift more, matching the Week 4 signal checks.

`ctr_gap` has about zero importance. Everything it knows is already covered by `ctr_h1` and `avg_pos_h1`. Nothing looks *suspiciously* perfect: no single feature gives near-1.0 AUC, unlike the deliberate `imp_h2` leak in Week 3.

### Where is it wrong? Error rates by group (test clients, RF top 500)

In [8]:
test = test.assign(rf_score=test_scores["Random Forest"])
test["rf_rank"] = test["rf_score"].rank(ascending=False, method="first")
top500 = test[test["rf_rank"] <= 500]

def by(col, bins=None, labels=None):
    g = pd.cut(test[col], bins, labels=labels) if bins is not None else test[col]
    t = test.assign(grp=g).groupby("grp", observed=True).agg(
        n=("is_declining", "size"), decline_rate=("is_declining", "mean"), avg_rf_score=("rf_score", "mean"))
    return t.round(3)

print("Top-500 precision:", round(top500["is_declining"].mean(), 3))
print("\nBy momentum (week-2 vs week-1 impressions):")
print(by("momentum_h1", [-0.01, 0.5, 0.9, 1.1, 2, 10], ["<0.5 falling", "0.5-0.9", "0.9-1.1 flat", "1.1-2", ">2 spiking"]).to_string())
print("\nBy position bucket:")
print(by("pos_bucket").to_string())
print("\nBy content type:")
print(by("content_type").to_string())

Top-500 precision: 0.89

By momentum (week-2 vs week-1 impressions):
                 n  decline_rate  avg_rf_score
grp                                           
<0.5 falling  3454         0.779         0.599
0.5-0.9       9762         0.491         0.364
0.9-1.1 flat  3774         0.302         0.250
1.1-2         4336         0.257         0.237
>2 spiking    3560         0.093         0.126

By position bucket:
          n  decline_rate  avg_rf_score
grp                                    
1-3    4724         0.340         0.289
3-5    6263         0.378         0.310
5-10   8142         0.469         0.362
10-20  3082         0.428         0.319
20+    2674         0.357         0.303

By content type:
                        n  decline_rate  avg_rf_score
grp                                                  
comparison article      2         0.000         0.480
feedly article         44         0.523         0.357
keyword article     24840         0.404         0.323


**What the errors look like:**
- **The model is basically a momentum ranker.** 78% of already-falling pages (<0.5) really decline and the model scores them highest. Spiking pages (>2) almost never decline (9%) and it scores them lowest. The hard, uncertain middle is the **flat pages (0.9–1.1)**: 30% decline, but the model gives them low scores, so most of those decliners are missed.
- **Position 5–10 has the most decliners (47%)**, and the model's average score is highest there too. So it follows the pattern, but pages 5–10 are where most of its misses live, simply because there are so many decliners.
- **Content type:** only 46 non-keyword pages are in the test clients, too few to judge. The model is effectively tested on keyword articles only.

### Three concrete wrong cases

In [9]:
show = ["content_hash_id", "rf_score", "is_declining", "imp_h1", "ctr_h1", "avg_pos_h1", "active_days_h1", "momentum_h1", "age_days"]

fp = test[(test["is_declining"] == 0)].nlargest(2, "rf_score")                      # confident, but did NOT decline
fn = test[(test["is_declining"] == 1) & (test["imp_h1"] >= 1000)].nsmallest(1, "rf_score")  # big page it missed
wrong = pd.concat([fp.assign(error="false alarm"), fn.assign(error="missed decline")])
wrong["content_hash_id"] = "…" + wrong["content_hash_id"].str[-6:]
wrong[["error"] + show].round(3)

,error,content_hash_id,rf_score,is_declining,imp_h1,ctr_h1,avg_pos_h1,active_days_h1,momentum_h1,age_days
37938,false alarm,…672455,0.868,0,94.0,0.000,8.021,11,0.096,233
69446,false alarm,…d0c958,0.858,0,67.0,0.000,5.403,7,0.000,31
6734,missed decline,…d01388,0.008,1,1004.0,0.013,1.453,5,10.000,11


**Why these three are hard:**
- **Both false alarms are small pages (67–94 impressions)** that nearly vanished in week 2 (momentum 0.0–0.1), so the model was very sure they'd keep falling. Instead they came back in the second half. On small pages, one quiet week is often just noise. **A higher minimum-impressions floor or a "small page" discount would cut these.**
- **The missed decline is an 11-day-old page** that was only live for 5 of the 15 days and was ramping up (momentum capped at 10). To the model, "rising fast" means safe, but a brand-new page's launch spike can fade just as fast. **New pages (under 30 days) should probably be scored separately or excluded.**

**Bottom line (decision-support, not a guarantee):** on clients it has never seen, the top 50 by momentum is right about 93% of the time on average (5-fold), and the Random Forest's about 88%, against about 39% for my Week-4 rule and ~30–40% for random picking. It's measured on one month (March 2026) of anonymized data, with a 2-week label that is noisy for small pages. The sealed final month is the next honest test. Seeds are fixed (42) and rows are sorted before splitting, so rerunning reproduces this table.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.